In [ ]:
# LIBRARIES

from langchain_openai import ChatOpenAI
from langchain.chains import create_sql_query_chain
from langchain_community.utilities import SQLDatabase

import pandas as pd
import sqlalchemy as sql
import os
import yaml
import re
from pprint import pprint

In [ ]:
# AI SETUP

os.environ["OPENAI_API_KEY"] = yaml.safe_load(open('../credentials.yml'))['openai']

OPENAI_LLM = "gpt-4o-mini"

In [ ]:
# DATABASE SETUP

PATH_DB = "sqlite:///database/leads_scored.db"

sql_engine = sql.create_engine(PATH_DB)

conn = sql_engine.connect()

In [ ]:
# 1.0 CREATE A SIMPLE SQL DATABASE AI AGENT

# * Connecting Langchain to a database

db = SQLDatabase.from_uri(PATH_DB)

In [ ]:
db.dialect

'sqlite'

In [ ]:
db.get_usable_table_names()

['leads', 'leads_scored', 'message_store', 'products', 'transactions']

In [ ]:
db.run("SELECT * FROM leads_scored LIMIT 10;")

"[(3, 'Garrick Langworth', 'garrick.langworth@gmail.com', 2, '2019-05-22 00:00:00.000000', 'in', 1, -589, 'gmail.com', 0, 0.9669123888015748, 0.0330875962972641), (4, 'Cordell Dickens', 'cordell.dickens@gmail.com', 4, '2018-11-19 00:00:00.000000', 'other', 1, -773, 'gmail.com', 0, 0.8558371067047119, 0.1441629081964492), (8, 'Inga Dach', 'inga.dach@gmail.com', 2, '2018-11-19 00:00:00.000000', 'other', 1, -773, 'gmail.com', 0, 0.9722884297370912, 0.0277115646749734), (10, 'Ferdinand Bergstrom', 'ferdinand.bergstrom@gmail.com', 2, '2020-03-20 00:00:00.000000', 'co', 1, -286, 'gmail.com', 0, 0.9663772583007812, 0.033622745424509), (11, 'Justen Simonis', 'justen.simonis@gmail.com', 2, '2020-04-14 00:00:00.000000', 'other', 0, -261, 'gmail.com', 0, 0.9866284132003784, 0.0133715886622667), (14, 'Marion Corwin', 'marion.corwin@hotmail.com', 2, '2019-11-04 00:00:00.000000', 'mx', 0, -423, 'hotmail.com', 0, 0.987798810005188, 0.0122011601924896), (18, 'Dr Sharif Kunde', 'dr.sharif.kunde@gmail.c

In [ ]:
# * Generating SQL with LLMs

model = ChatOpenAI(
    model = OPENAI_LLM,
    temperature = 0.7,
)

chain = create_sql_query_chain(model, db)

chain

RunnableAssign(mapper={
  input: RunnableLambda(...),
  table_info: RunnableLambda(...)
})
| RunnableLambda(lambda x: {k: v for (k, v) in x.items() if k not in ('question', 'table_names_to_use')})
| PromptTemplate(input_variables=['input', 'table_info'], partial_variables={'top_k': '5'}, template='You are a SQLite expert. Given an input question, first create a syntactically correct SQLite query to run, then look at the results of the query and return the answer to the input question.\nUnless the user specifies in the question a specific number of examples to obtain, query for at most {top_k} results using the LIMIT clause as per SQLite. You can order the results to return the most informative data in the database.\nNever query for all columns from a table. You must query only the columns that are needed to answer the question. Wrap each column name in double quotes (") to denote them as delimited identifiers.\nPay attention to use only the column names you can see in the tables below.

In [ ]:
response = chain.invoke({'question': "which 5 customers have the highest p1 probability of purchase?"})

In [ ]:
pprint(response)

('```sql\n'
 'SELECT "user_full_name", "user_email", "p1" \n'
 'FROM leads_scored \n'
 'ORDER BY "p1" DESC \n'
 'LIMIT 5;\n'
 '```')


In [ ]:
pprint(db.run(response))

OperationalError: (sqlite3.OperationalError) near "```sql
SELECT "user_full_name", "user_email", "p1" 
FROM leads_scored 
ORDER BY "p1" DESC 
LIMIT 5;
```": syntax error
[SQL: ```sql
SELECT "user_full_name", "user_email", "p1" 
FROM leads_scored 
ORDER BY "p1" DESC 
LIMIT 5;
```]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
# * Parsing SQL

def extract_sql_code(text):
    sql_code_match = re.search(r'```sql(.*?)```', text, re.DOTALL)
    if sql_code_match:
        sql_code = sql_code_match.group(1).strip()
        return sql_code
    else:
        sql_code_match = re.search(r"sql(.*?)'", text, re.DOTALL)
        if sql_code_match:
            sql_code = sql_code_match.group(1).strip()
            return sql_code
        else:
            return None

pprint(extract_sql_code(response))

('SELECT "user_full_name", "user_email", "p1" \n'
 'FROM leads_scored \n'
 'ORDER BY "p1" DESC \n'
 'LIMIT 5;')


In [ ]:
pprint(db.run(extract_sql_code(response)))

("[('Kecia Johnson', 'kecia.johnson@gmail.com', 0.9326744079589844), ('Estel "
 "Abernathy', 'estel.abernathy@gmail.com', 0.9326744079589844), ('Peter "
 "Morar-Lebsack', 'peter.morarlebsack@gmail.com', 0.9239947199821472), ('Dr. "
 "Alana Hermann DDS', 'dr.alana.hermann.dds@gmail.com', 0.9134250283241272), "
 "('Bedford Farrell', 'bedford.farrell@yahoo.com', 0.8758423924446106)]")


In [ ]:
# 2.0 CREATE A PANDAS SQL DATABASE AGENT

# * Working with Pandas

In [ ]:
response = chain.invoke({'question': "which 5 customers have the highest p1 probability of purchase but have not yet purchased anything?"})

pd.read_sql(extract_sql_code(response), conn)

,user_full_name,user_email,p1
0,Kamryn Tremblay,kamryn.tremblay@me.com,0.759711
1,Mr Dewitt Conroy IV,mr.dewitt.conroy.iv@gmail.com,0.628244
2,Mr. Jereme Johnston DDS,mr.jereme.johnston.dds@gmail.com,0.622120
3,Mr Ignacio Braun I,mr.ignacio.braun.i@gmail.com,0.620640
4,Lavonia Krajcik,lavonia.krajcik@gmail.com,0.614534


In [ ]:
conn.close()